# Global histogram — grayscale values in masks (JPEG raw)

Scans files in `masks/` (as in training: `np.fromfile` + `cv2.imdecode` in grayscale) and accumulates un histograma **0…255** over **all pixels**.

- By default: only `image_name` listed in `partition/` (same logic as `analyze_masks.py` without `--all-masks`).
- Optional: `USE_ALL_MASKS_IN_FOLDER = True` to use all files in `masks/`.
- Optional: `MAX_MASKS` to limit the number of masks (quick test).

**Kernel:** project environment with `cv2`, `matplotlib`, `pandas`.

In [1]:
from __future__ import annotations

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


def find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "partition").is_dir() and (p / "masks").is_dir():
            return p
    raise FileNotFoundError("Could not find project with partition/ and masks/; set BASE manually.")


BASE = find_project_root()
MASKS_DIR = BASE / "masks"
PARTITION = BASE / "partition"

# --- Options ---
USE_ALL_MASKS_IN_FOLDER = False  # True → ignore partition, use all .jpg/.png in masks/
MAX_MASKS: int | None = None  # p. ej. 500 for testing; None = no limit

print("BASE:", BASE)
print("MASKS_DIR:", MASKS_DIR)

BASE: c:\Users\Aleix\OneDrive - Universitat Politècnica de Catalunya\Escritorio\UNI\TFG\Recerca primers datasets\SicapV2\SICAPv2
MASKS_DIR: c:\Users\Aleix\OneDrive - Universitat Politècnica de Catalunya\Escritorio\UNI\TFG\Recerca primers datasets\SicapV2\SICAPv2\masks


In [3]:
def list_partition_image_names() -> set[str]:
    patterns = (
        "Validation/*/Train.xlsx",
        "Validation/*/Test.xlsx",
        "Test/Train.xlsx",
        "Test/Test.xlsx",
    )
    names: set[str] = set()
    for pat in patterns:
        for xp in sorted(PARTITION.glob(pat)):
            df = pd.read_excel(xp)
            if "image_name" not in df.columns:
                continue
            for v in df["image_name"].dropna():
                s = str(v).strip()
                if s:
                    names.add(s)
    return names


def collect_mask_paths(*, all_masks: bool) -> list[Path]:
    if all_masks:
        paths = sorted(MASKS_DIR.glob("*.jpg")) + sorted(MASKS_DIR.glob("*.png"))
        return paths
    allowed = list_partition_image_names()
    if not allowed:
        raise RuntimeError(f"No image_name found en {PARTITION}")
    out: list[Path] = []
    missing = 0
    for name in sorted(allowed):
        p = MASKS_DIR / name
        if p.is_file():
            out.append(p)
        else:
            missing += 1
    if missing:
        print(f"Warning: {missing} nombres in partition sin file in masks/")
    return out


mask_paths = collect_mask_paths(all_masks=USE_ALL_MASKS_IN_FOLDER)
if MAX_MASKS is not None:
    mask_paths = mask_paths[: int(MAX_MASKS)]
print(f"Masks a leer: {len(mask_paths):,}")

Masks a leer: 12,081


In [ ]:
def read_mask_gray(path: Path) -> np.ndarray | None:
    buf = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)


def histogram_all_pixels(paths: list[Path]) -> np.ndarray:
  """Sum of bincount 0..255 over all pixels from all masks."""
    def histogram_all_pixels(paths: list[Path]) -> np.ndarray:
        """Sum of bincount 0..255 over all pixels from all masks."""
        h = np.zeros(256, dtype=np.float64)
        n_ok = 0
        n_fail = 0
        for p in tqdm(paths, desc="Masks"):
            m = read_mask_gray(p)
            if m is None:
                n_fail += 1
                continue
            n_ok += 1
            bc = np.bincount(m.ravel(), minlength=256)
            h += bc.astype(np.float64)
        print(f"Read OK: {n_ok:,} | failed (imdecode None): {n_fail:,}")
        return h
    n_ok = 0
    n_fail = 0
    for p in tqdm(paths, desc="Masks"):
        m = read_mask_gray(p)
        if m is None:
            n_fail += 1
            continue
        n_ok += 1
        bc = np.bincount(m.ravel(), minlength=256)
        h += bc.astype(np.float64)
    print(f"Read OK: {n_ok:,} | failed (imdecode None): {n_fail:,}")
    return h


hist = histogram_all_pixels(mask_paths)
total_px = float(hist.sum())
print(f"Total accumulated pixels: {total_px:,.0f}")

IndentationError: unexpected indent (2419350474.py, line 8)

In [ ]:
x = np.arange(256)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(x, hist, width=1.0, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Grayscale value (decoded mask)")
axes[0].set_ylabel("Pixel count")
axes[0].set_title("Global histogram (linear)")
axes[0].set_xlim(-0.5, 255.5)

axes[1].bar(x, np.log1p(hist), width=1.0, color="coral", edgecolor="none")
axes[1].set_xlabel("Grayscale value")
axes[1].set_ylabel("log(1 + count)")
axes[1].set_title("Same distribution (log scale on Y-axis to inspect tails)")
axes[1].set_xlim(-0.5, 255.5)

mode = "all in masks/" if USE_ALL_MASKS_IN_FOLDER else "partition only"
fig.suptitle(f"Original masks — {mode} — N={len(mask_paths):,} files")
plt.tight_layout()
plt.show()